In [2]:
import sympy as sp
from sympy.physics.quantum import Commutator, Dagger, Operator

Goal: Check the 2LS math in the freespace document (mainly simplification of the SLH expression for X(t))

$$dX(t)=-i[X(t),H(t)]dt + \mathcal{L}^\dagger[L(t)]X(t)dt + [L^\dagger(t), X(t)]dB_{in}[t] + [X(t), L(t)] dB_{in}^\dagger[t]$$

Becomes: 

\begin{align}
\frac{dX(t)}{dt}=-i[X(t),H(t)] + \mathcal{L}^\dagger[L(t)]X(t) + [L^\dagger(t), X(t)]\frac{dB_{in}[t]}{dt} + [X(t), L(t)] \frac{dB_{in}^\dagger[t]}{dt}
\end{align}

- $\mathcal{L}^{\dagger}[L] X=L^{\dagger} X L-\frac{1}{2}\left(L^{\dagger} L X+X L^{\dagger} L\right) $
- Why does $\frac{dB_{in}[t]}{dt}$ become $\hat{a}_{in}$?

In [27]:
# Define sigma_+ and sigma_- as symbolic operators
sigma_plus = Operator('\sigma_+')
sigma_minus = Operator('\sigma_-')
sigma_z = Operator('\sigma_z')

t = sp.Symbol('t')
g = sp.Function('g')(t)
a_in = sp.Function('a_{in}')(t)

def lindblad_operator(X, L):
    return Dagger(L)*X*L - 1/2*(Dagger(L)*L*X + X*Dagger(L)*L)

def get_expr(X, H, L, a_in, t):
    term1 = -1j*Commutator(X, H).doit()
    term2 = lindblad_operator(X,L)
    term3 = Commutator(Dagger(L), X).doit() * a_in # sp.diff(a_in, t)
    term4 = Commutator(X, L).doit() * Dagger(a_in) # Dagger(sp.diff(a_in, t))
    expr = term1 + term2 + term3 + term4
    return expr

H = 0 
S = sp.eye(2)
L = -Dagger(g)*sigma_minus
X = sigma_plus
expr = get_expr(X, H, L, a_in, t)

In [28]:
expr

-a_{in}(t)*g(t)*(Dagger(\sigma_-)*\sigma_+ - \sigma_+*Dagger(\sigma_-)) + g(t)*Dagger(g(t))*Dagger(\sigma_-)*\sigma_+*\sigma_- - Dagger(a_{in}(t))*Dagger(g(t))*(\sigma_+*\sigma_- - \sigma_-*\sigma_+) - 0.5*(g(t)*Dagger(g(t))*Dagger(\sigma_-)*\sigma_-*\sigma_+ + g(t)*Dagger(g(t))*\sigma_+*Dagger(\sigma_-)*\sigma_-)

In [29]:
g_mag_sq = sp.Symbol('|g|^2', real=True, positive=True)

# Substitute values and veirfy correctness of substitutions (using matrices)
sigma_p_val = sp.Matrix([[0, 1], [0, 0]])
sigma_m_val = sp.Matrix([[0, 0], [1, 0]])
sigma_z_val = sp.Matrix([[1, 0], [0, -1]])

expr = expr.subs(g*Dagger(g), g_mag_sq)
expr = expr.subs(Dagger(sigma_minus), sigma_plus)
expr = expr.subs(Dagger(sigma_plus), sigma_minus)

print(sigma_p_val@sigma_p_val == sp.eye(2)*0)
expr = expr.subs(sigma_plus*sigma_plus, 0)

print(sigma_p_val@sigma_m_val - sigma_m_val@sigma_p_val == sigma_z_val)
expr = expr.subs(sigma_plus*sigma_minus - sigma_minus*sigma_plus, sigma_z)

print(sigma_p_val@sigma_m_val@sigma_p_val == sigma_p_val)
expr = expr.subs(sigma_plus*sigma_minus*sigma_plus, sigma_plus)

True
True
True


In [30]:
expr

-0.5*|g|^2*\sigma_+ - Dagger(a_{in}(t))*Dagger(g(t))*\sigma_z

# $\sigma_-$

In [31]:
X = sigma_minus
expr = get_expr(X, H, L, a_in, t)
expr

-a_{in}(t)*g(t)*(Dagger(\sigma_-)*\sigma_- - \sigma_-*Dagger(\sigma_-)) + g(t)*Dagger(g(t))*Dagger(\sigma_-)*\sigma_-**2 - 0.5*(g(t)*Dagger(g(t))*Dagger(\sigma_-)*\sigma_-**2 + g(t)*Dagger(g(t))*\sigma_-*Dagger(\sigma_-)*\sigma_-)

In [32]:
expr = expr.subs(g*Dagger(g), g_mag_sq)
expr = expr.subs(Dagger(sigma_minus), sigma_plus)
expr = expr.subs(Dagger(sigma_plus), sigma_minus)

print(sigma_p_val@sigma_p_val == sp.eye(2)*0)
expr = expr.subs(sigma_plus*sigma_plus, 0)

print(sigma_m_val@sigma_m_val == sp.eye(2)*0)
expr = expr.subs(sigma_minus*sigma_minus, 0)

print(sigma_p_val@sigma_m_val - sigma_m_val@sigma_p_val == sigma_z_val)
expr = expr.subs(sigma_plus*sigma_minus - sigma_minus*sigma_plus, sigma_z)

print(sigma_p_val@sigma_m_val@sigma_p_val == sigma_p_val)
expr = expr.subs(sigma_plus*sigma_minus*sigma_plus, sigma_plus)

print(sigma_m_val@sigma_p_val@sigma_m_val == sigma_m_val)
expr = expr.subs(sigma_minus*sigma_plus*sigma_minus, sigma_minus)

True
True
True
True
True


In [33]:
expr

-0.5*|g|^2*\sigma_- - a_{in}(t)*g(t)*\sigma_z

# $\sigma_z$

In [57]:
X = sigma_z
expr = get_expr(X, H, L, a_in, t)
expr = expr.subs(g*Dagger(g), g_mag_sq)
expr = expr.subs(Dagger(sigma_minus), sigma_plus)
expr = expr.subs(Dagger(sigma_plus), sigma_minus)
expr

|g|^2*\sigma_+*\sigma_z*\sigma_- - a_{in}(t)*g(t)*(\sigma_+*\sigma_z - \sigma_z*\sigma_+) + Dagger(a_{in}(t))*Dagger(g(t))*(\sigma_-*\sigma_z - \sigma_z*\sigma_-) - 0.5*(|g|^2*\sigma_+*\sigma_-*\sigma_z + |g|^2*\sigma_z*\sigma_+*\sigma_-)

In [58]:
I = Operator('I')
sigma_ee = (I + sigma_z)/2
sigma_gg = (I - sigma_z)/2

sigma_ee_val = sp.Matrix([[1, 0], [0, 0]])
print((sp.eye(2) + sigma_z_val) / 2 == sigma_ee_val)

True


In [59]:
print(sigma_p_val*sigma_m_val*sigma_z_val == sigma_ee_val)
expr = expr.subs(sigma_plus*sigma_minus*sigma_z, sigma_ee)

print(sigma_z_val*sigma_p_val*sigma_m_val == sigma_ee_val)
expr = expr.subs(sigma_z*sigma_plus*sigma_minus, sigma_ee)

print(sigma_p_val*sigma_z_val*sigma_m_val == -sigma_ee_val)
expr = expr.subs(sigma_plus*sigma_z*sigma_minus, -sigma_ee)

print(sigma_p_val@sigma_z_val - sigma_z_val@sigma_p_val == -2*sigma_p_val)
expr = expr.subs(sigma_plus*sigma_z - sigma_z*sigma_plus, -2*sigma_plus)

sigma_m_val@sigma_z_val - sigma_z_val@sigma_m_val == 2*sigma_m_val
expr = expr.subs(sigma_minus*sigma_z - sigma_z*sigma_minus, 2*sigma_minus)

True
True
True
True


In [60]:
expr = sp.simplify(expr)
expr

-1.0*|g|^2*(I + \sigma_z) + 2*a_{in}(t)*g(t)*\sigma_+ + 2*Dagger(a_{in}(t))*Dagger(g(t))*\sigma_-

In [56]:
sp.expand(0.5*expr)

-0.5*|g|^2*I - 0.5*|g|^2*\sigma_z + 1.0*a_{in}(t)*g(t)*\sigma_+ + 1.0*Dagger(a_{in}(t))*Dagger(g(t))*\sigma_-